# Notebook 5. CBOW Word Embeddings From Scratch (PyTorch) - SOLUTION

*ML & NLP course - Data Trainers LLC - Axel Sirota*

## The story

In Module 1 our logistic-regression classifier saw words as one-hot columns: `king` and `queen` sat on orthogonal axes, `dog` was as far from `canine` as from `car`. This notebook lifts that ceiling by learning **dense word vectors** from Yelp reviews using **Continuous Bag of Words (CBOW)**.

By the end you have:

1. A 100-dimensional vector for every vocabulary word.
2. Confirmation that semantically related words cluster together.
3. A PyTorch training loop you wrote yourself.

## Section 0. Environment Setup

In [ ]:
# Install required packages (Colab)
!pip install -q torch gensim textblob scikit-learn matplotlib pandas numpy
!python -m textblob.download_corpora

In [ ]:
# Core imports - visualization -> data -> model -> training
import random
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from textblob import TextBlob
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# Hyperparameters
SEED = 42
CORPUS_SIZE = 5000
VOCAB_MIN_FREQ = 5
WINDOW_SIZE = 2
EMBEDDING_DIM = 100
BATCH_SIZE = 128
EPOCHS = 5
LR = 1e-3

# Seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete.")

## Section 1. Why Embeddings?

### The one-hot problem

Cosine similarity between any two distinct one-hot vectors is exactly 0, so every pair of different words looks equally unrelated.

In [ ]:
# Demo: one-hot vectors are mutually orthogonal
vocab_toy = ['king', 'queen', 'dog', 'car']
V = len(vocab_toy)
one_hot = np.eye(V)
sims = cosine_similarity(one_hot)
sim_df = pd.DataFrame(sims, index=vocab_toy, columns=vocab_toy)
print("Cosine similarity of one-hot vectors:")
print(sim_df)
print("\nNote: every off-diagonal entry is 0.")

### The distributional hypothesis

*"You shall know a word by the company it keeps."* - J. R. Firth, 1957.

**CBOW** turns this into an algorithm: predict the center word from its surrounding context. The network can only succeed if it learns dense vectors that encode contextual similarity.

## Section 2. Building the Dataset

In [ ]:
# Load Yelp reviews from Dropbox
URL = 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(URL)
print(f"Full dataset: {len(yelp):,} reviews")

yelp_sample = yelp.sample(n=CORPUS_SIZE, random_state=SEED).reset_index(drop=True)
print(f"Subsampled for CBOW: {len(yelp_sample):,} reviews")

yelp_sample[['stars', 'text']].head(3)

In [ ]:
# Demo: tokenize a review with TextBlob
example = yelp_sample.iloc[0]['text']
print("Raw review:")
print(example[:200], '...')

tokens_example = [w.lower() for w in TextBlob(example).words]
print(f"\nFirst 20 tokens: {tokens_example[:20]}")
print(f"Token count: {len(tokens_example)}")

In [ ]:
# Tokenize ALL reviews
def tokenize(text):
    return [w.lower() for w in TextBlob(str(text)).words]

tokenized_reviews = [tokenize(t) for t in yelp_sample['text']]
print(f"Tokenized {len(tokenized_reviews):,} reviews")
print(f"Avg tokens/review: {np.mean([len(r) for r in tokenized_reviews]):.1f}")

all_tokens = [tok for review in tokenized_reviews for tok in review]
print(f"Total tokens: {len(all_tokens):,}")

In [ ]:
# Build vocab with min_freq cutoff. id 0 is reserved for <UNK>.
def build_vocab(tokens, min_freq=5):
    counts = Counter(tokens)
    kept = [w for w, c in counts.most_common() if c >= min_freq]
    word2id = {'<UNK>': 0}
    for w in kept:
        word2id[w] = len(word2id)
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

word2id, id2word = build_vocab(all_tokens, min_freq=VOCAB_MIN_FREQ)
vocab_size = len(word2id)
print(f"Vocabulary size (min_freq={VOCAB_MIN_FREQ}): {vocab_size:,}")
print(f"Sample mappings: {list(word2id.items())[:10]}")

In [ ]:
# Encode every token (unknown -> 0)
UNK = word2id['<UNK>']

def encode(tokens):
    return [word2id.get(t, UNK) for t in tokens]

encoded_reviews = [encode(r) for r in tokenized_reviews]
print("Tokens:", tokenized_reviews[0][:10])
print("IDs:   ", encoded_reviews[0][:10])

### The CBOW window

With `WINDOW_SIZE = 2`, for each center word we collect the 2 words on each side as context.

In [ ]:
# Demo: build (context, target) pairs
def make_context_target_pairs(ids, window=2):
    pairs = []
    for i in range(window, len(ids) - window):
        context = ids[i - window:i] + ids[i + 1:i + window + 1]
        target = ids[i]
        pairs.append((context, target))
    return pairs

toy_ids = encoded_reviews[0][:10]
toy_words = [id2word.get(i, '<UNK>') for i in toy_ids]
print("Tokens:", toy_words)
print("IDs:   ", toy_ids)

toy_pairs = make_context_target_pairs(toy_ids, window=WINDOW_SIZE)
print("\n(context, target) pairs:")
for ctx, tgt in toy_pairs:
    print(f"  {[id2word[i] for i in ctx]}  ->  {id2word[tgt]}")

### Lab 1. Build the `CBOWDataset` (SOLUTION)

Wraps the pair-generation logic in a PyTorch `Dataset` and a `DataLoader`.

In [ ]:
# Solution: CBOWDataset

class CBOWDataset(Dataset):
    def __init__(self, encoded_reviews, window=2):
        # 1. Flatten all (context, target) pairs from every review into one big list.
        #    A list comprehension is the cleanest way; for big corpora consider
        #    using a generator + np.array to save RAM.
        self.pairs = []
        for review in encoded_reviews:
            self.pairs.extend(make_context_target_pairs(review, window))
        self.window = window

    def __len__(self):
        # 2. Total number of training pairs
        return len(self.pairs)

    def __getitem__(self, idx):
        # 3. Convert the stored Python lists/ints into torch.long tensors.
        #    DataLoader will batch these for us by stacking dim 0.
        context, target = self.pairs[idx]
        context_tensor = torch.tensor(context, dtype=torch.long)   # shape (2*window,)
        target_tensor = torch.tensor(target, dtype=torch.long)     # 0-d scalar
        return context_tensor, target_tensor


# 4. Build dataset and DataLoader.
#    num_workers=0 is intentional: more workers can hang on Colab kernels.
dataset = CBOWDataset(encoded_reviews, window=WINDOW_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Total training pairs: {len(dataset):,}")
ctx_batch, tgt_batch = next(iter(loader))
print(f"Context batch shape: {ctx_batch.shape}  dtype={ctx_batch.dtype}")
print(f"Target batch shape:  {tgt_batch.shape}  dtype={tgt_batch.dtype}")

# We store pairs in a flat list (instead of generating them on the fly) so
# that __getitem__ is O(1): DataLoader workers call it constantly. For very
# large corpora you would swap to a generator-based IterableDataset, but for
# 5K reviews a flat list fits comfortably in RAM.

## Section 3. The CBOW Model

Architecture:

```
(B, 4)  --nn.Embedding(V, D)-->  (B, 4, D)
        --mean over dim=1 ------>  (B, D)
        --nn.Linear(D, V) ------>  (B, V)   (logits - NO softmax in forward)
```

In [ ]:
# Demo: walk through shapes
dummy_V = 100
dummy_D = 8
dummy_emb = nn.Embedding(dummy_V, dummy_D)
dummy_lin = nn.Linear(dummy_D, dummy_V)

fake_ctx = torch.randint(0, dummy_V, (3, 4))
print("Input ids shape:        ", fake_ctx.shape)

e = dummy_emb(fake_ctx)
print("After Embedding:         ", e.shape, "  # (B, 2*window, D)")

pooled = e.mean(dim=1)
print("After mean over dim=1:   ", pooled.shape, "  # (B, D)")

logits = dummy_lin(pooled)
print("After Linear:            ", logits.shape, "  # (B, V)")

print("\nEmbedding weight matrix shape:", dummy_emb.weight.shape)

### Lab 2. Implement the CBOW model (SOLUTION)

In [ ]:
# Solution: CBOW model

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # 1. Embedding lookup table: V rows, D columns.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 2. Linear projection from the pooled embedding back to vocab logits.
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_ids):
        # context_ids: (B, 2*window) of long ids
        emb = self.embedding(context_ids)    # (B, 2*window, D)
        pooled = emb.mean(dim=1)             # (B, D)  - order-invariant pooling
        logits = self.linear(pooled)         # (B, V)  - raw logits
        # Do not apply softmax here: nn.CrossEntropyLoss does it for us
        # (log_softmax + NLLLoss fused for numerical stability).
        return logits


model = CBOW(vocab_size, EMBEDDING_DIM).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

# Sanity check forward pass
dummy = torch.randint(0, vocab_size, (2, 2 * WINDOW_SIZE), device=device)
out = model(dummy)
print(f"Output logits shape: {out.shape}  (expected: (2, {vocab_size}))")

# The embedding matrix dominates the param count: V * D parameters. The
# Linear layer adds another (D + 1) * V (the +1 is the bias). Weight tying
# (reusing the embedding matrix as the output projection) would halve the
# params; Word2Vec does not do this, but some modern models do.

## Section 4. The Training Loop

Six lines per batch:

```
1. move tensors to device
2. optimizer.zero_grad()    <-- never forget this
3. forward
4. loss
5. loss.backward()
6. optimizer.step()
```

Sanity-check the **initial loss**: a random model should give cross-entropy ≈ `ln(vocab_size)`.

In [ ]:
# Set up loss + optimizer, verify initial loss
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

expected = np.log(vocab_size)
print(f"Expected initial loss ~ ln({vocab_size}) = {expected:.3f}")

ctx, tgt = next(iter(loader))
ctx, tgt = ctx.to(device), tgt.to(device)
with torch.no_grad():
    init_logits = model(ctx)
    init_loss = loss_fn(init_logits, tgt)
print(f"Actual initial loss: {init_loss.item():.3f}")
# If these are wildly different, your setup is wrong (e.g., softmax in forward,
# wrong loss, mis-shaped logits).

In [ ]:
# Single backward pass - confirm autograd populates gradients
optimizer.zero_grad()
logits = model(ctx)
loss = loss_fn(logits, tgt)
loss.backward()
print("Gradient norm on embedding matrix:",
      model.embedding.weight.grad.norm().item())
print("Gradient shape:", model.embedding.weight.grad.shape)

### Lab 3. Write the training loop (SOLUTION)

In [ ]:
# Solution: full training loop

# Fresh optimizer so re-running this cell starts clean
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

epoch_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for context, target in loader:
        # 1. Move to device - every tensor that touches the model must live there
        context = context.to(device)
        target = target.to(device)

        # 2. Clear gradients from the previous step.
        #    PyTorch accumulates gradients by default, so skipping this is a
        #    common source of silent bugs.
        optimizer.zero_grad()

        # 3. Forward pass - gives logits of shape (B, V)
        logits = model(context)

        # 4. Compute the loss. CrossEntropyLoss expects:
        #    - logits: (B, V) raw scores
        #    - target: (B,) class indices (NOT one-hot)
        loss = loss_fn(logits, target)

        # 5. Backprop: builds .grad on every parameter that contributed
        loss.backward()

        # 6. Apply the gradient update via Adam
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    avg = total_loss / max(n_batches, 1)
    epoch_losses.append(avg)
    print(f"Epoch {epoch + 1}/{EPOCHS} - avg loss: {avg:.4f}")

# Notes:
# - model.train() at the start of each epoch is a no-op for CBOW, but it
#   matters once you add Dropout or BatchNorm.
# - loss.item() pulls a Python float off the tensor and implicitly syncs
#   GPU work, so call it sparingly inside hot loops if performance matters.
# - Typical issues to watch for: forgetting optimizer.zero_grad() causes
#   gradients to sum across batches; forgetting .to(device) on the target
#   raises a CUDA assertion error; applying softmax in forward combined
#   with CrossEntropyLoss applies softmax twice and kills gradients.

In [ ]:
# Plot loss curve
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Avg training loss')
plt.title('CBOW training loss')
plt.grid(True, alpha=0.3)
plt.show()

### Diagnostics

1. Loss decreasing? Good.
2. Any NaNs? Lower the LR or add gradient clipping.
3. Val/train divergence? Overfitting (not a concern at this scale).

In [ ]:
# Save model state
torch.save(model.state_dict(), 'cbow.pt')
print("Saved cbow.pt")

## Section 5. Using the Embeddings

In [ ]:
# Extract embeddings and wrap in gensim KeyedVectors
from gensim.models import KeyedVectors

embedding_matrix = model.embedding.weight.detach().cpu().numpy()
print(f"Embedding matrix shape: {embedding_matrix.shape}")

kv = KeyedVectors(vector_size=EMBEDDING_DIM)
words = [id2word[i] for i in range(vocab_size)]
kv.add_vectors(words, embedding_matrix)
print(f"KeyedVectors contains {len(kv):,} words")

print("\nWords closest to 'good':")
for w, sim in kv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={sim:.3f}")

### Lab 4. Probe the embedding space (SOLUTION)

In [ ]:
# Solution: probe 5 anchor words
anchors = ['good', 'bad', 'service', 'food', 'expensive']

for anchor in anchors:
    # 1. Skip OOV anchors - gensim raises KeyError otherwise.
    if anchor not in kv:
        print(f"'{anchor}' not in vocab - skipping")
        continue
    # 2. Top-5 similar words
    neighbors = kv.most_similar(anchor, topn=5)
    print(f"\nTop-5 neighbors of '{anchor}':")
    for w, s in neighbors:
        print(f"  {w:20s}  cos={s:.3f}")

# 3. OBSERVATIONS (typical pattern on a 5K Yelp corpus):
#   - 'good':       neighbours mix synonyms (great, nice) with antonyms (bad, terrible)
#                   because all of them appear in the slot 'the food was ___'.
#                   This is the distributional-similarity vs. sentiment-similarity trap.
#   - 'bad':        symmetric - pulls in 'good', 'awful', 'horrible'.
#   - 'service':    'staff', 'waiter', 'waitress', 'manager' - cleaner cluster.
#   - 'food':       'meal', 'dinner', 'menu', 'cuisine' - restaurant-domain words.
#   - 'expensive':  'pricey', 'overpriced', 'cheap' - pricing-slot words, mixed sign.

### The famous analogy: `king - man + woman ≈ queen`

Almost certainly will NOT work on Yelp, because those words barely appear. Try it anyway as a teaching moment.

In [ ]:
# Demo: the analogy - expect weak results on Yelp
def try_analogy(kv, pos, neg, topn=5):
    try:
        return kv.most_similar(positive=pos, negative=neg, topn=topn)
    except KeyError as e:
        return f"OOV word in analogy: {e}"

print("king - man + woman ≈ ?")
print(try_analogy(kv, pos=['king', 'woman'], neg=['man']))

print("\ngood - great + bad ≈ ?  (domain analogy - more likely to work on Yelp)")
print(try_analogy(kv, pos=['good', 'bad'], neg=['great']))

### PCA visualization

In [ ]:
# Demo: PCA of the 50 most frequent words
freq_order = [id2word[i] for i in range(1, 51)]
freq_vecs = np.stack([kv[w] for w in freq_order])

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(freq_vecs)

plt.figure(figsize=(10, 7))
plt.scatter(pca_2d[:, 0], pca_2d[:, 1], s=30, alpha=0.6)
for (x, y), w in zip(pca_2d, freq_order):
    plt.text(x + 0.01, y + 0.01, w, fontsize=9)
plt.title('PCA of 50 most frequent words (CBOW-Yelp 100D -> 2D)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.show()

### Lab 5. Domain cluster plot (SOLUTION)

In [ ]:
# Solution: domain cluster plot

# 1. Word groups by domain
groups = {
    'food':     ['pizza', 'chicken', 'salad', 'burger', 'sandwich', 'taco', 'pasta'],
    'service':  ['waiter', 'staff', 'service', 'manager', 'host', 'waitress'],
    'ambience': ['music', 'decor', 'atmosphere', 'seating', 'patio', 'lighting'],
}

# 2. Filter out OOV words per group
filtered = {dom: [w for w in ws if w in kv] for dom, ws in groups.items()}
for dom, ws in filtered.items():
    dropped = set(groups[dom]) - set(ws)
    if dropped:
        print(f"  Dropped (OOV) from {dom}: {sorted(dropped)}")

# 3. Flatten to parallel lists + matrix
flat_words = [w for dom in filtered for w in filtered[dom]]
flat_domains = [dom for dom in filtered for _ in filtered[dom]]
flat_vecs = np.stack([kv[w] for w in flat_words])
print(f"Total words plotted: {len(flat_words)}")

# 4. PCA to 2D
coords_2d = PCA(n_components=2, random_state=SEED).fit_transform(flat_vecs)

# 5. Scatter, colored by domain
domain_colors = {'food': 'tab:red', 'service': 'tab:blue', 'ambience': 'tab:green'}
plt.figure(figsize=(10, 7))
for d, color in domain_colors.items():
    mask = np.array([dom == d for dom in flat_domains])
    if mask.any():
        plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], c=color, label=d, s=60)
for (x, y), w in zip(coords_2d, flat_words):
    plt.text(x + 0.01, y + 0.01, w, fontsize=9)
plt.legend(); plt.title('Domain clusters in CBOW-Yelp embedding space')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(True, alpha=0.3)
plt.show()

# 6. OBSERVATION: with only 5K reviews + 100D, separation is *partial*.
#    Food and service usually pull apart along PC1; ambience is the smallest
#    domain in this corpus and tends to overlap with food/service. PCA loses
#    98 dimensions of structure, so weak clustering in 2D doesn't mean the
#    100D vectors are bad. Try t-SNE or UMAP for a fairer visualisation.

### Limitations

Small corpus, domain-skewed vocab, no famous analogies, distributional ≠ sentiment. NB6 fixes most of this with pretrained GloVe.

## Section 6. Wrap-up

Compare with gensim's production CBOW.

In [ ]:
# Demo: gensim Word2Vec for reference
import time
from gensim.models import Word2Vec

t0 = time.time()
w2v_ref = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=EMBEDDING_DIM,
    window=WINDOW_SIZE,
    min_count=VOCAB_MIN_FREQ,
    sg=0,
    workers=2,
    epochs=EPOCHS,
    seed=SEED,
)
t_gensim = time.time() - t0
print(f"gensim Word2Vec trained in {t_gensim:.1f}s")

print("\nOur CBOW 'good':")
for w, s in kv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={s:.3f}")

print("\ngensim 'good':")
for w, s in w2v_ref.wv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={s:.3f}")

### Optional lab. Sweep embedding_dim (SOLUTION)

In [ ]:
# Solution: dim sweep

def train_cbow_quick(dim, epochs=EPOCHS):
    m = CBOW(vocab_size, dim).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    for ep in range(epochs):
        m.train()
        for ctx, tgt in loader:
            ctx, tgt = ctx.to(device), tgt.to(device)
            opt.zero_grad()
            logits = m(ctx)
            loss = loss_fn(logits, tgt)
            loss.backward()
            opt.step()
    return m

# Note: this re-trains 3 models - expect a few minutes on GPU, longer on CPU.
# Comment out if you want to skip.
for dim in [50, 100, 300]:
    m = train_cbow_quick(dim)
    mat = m.embedding.weight.detach().cpu().numpy()
    kv_d = KeyedVectors(vector_size=dim)
    kv_d.add_vectors(words, mat)
    print(f"\n--- dim={dim} --- most similar to 'food':")
    for w, s in kv_d.most_similar('food', topn=5):
        print(f"  {w:20s}  cos={s:.3f}")

# OBSERVATION: with only 5K reviews, dim=50 already saturates. dim=300 has
# more capacity than signal, so neighbours often look noisier, not sharper.
# This is the classic capacity-vs-data trade-off.

## Wrap-up

### What you learned

- **Distributional hypothesis**: meaning lives in context.
- **CBOW**: predict the center word from its surrounding context; the embedding matrix is the real output.
- **PyTorch workflow**: `Dataset`, `DataLoader`, `nn.Module`, `CrossEntropyLoss`, `Adam`, and the 6-line training loop.
- **Dense embeddings**: extracted from `model.embedding.weight` and queried via `KeyedVectors`.
- **Limitations of small corpora**: a 5K-review corpus yields a domain-skewed vocabulary and noisy neighbours, which motivates moving to pretrained GloVe in NB6.

### Self-check quiz answers

1. **Why mean-pool instead of concatenate?**
   - Mean-pooling produces a fixed-size `(D,)` vector regardless of window size, and is **order-invariant** (matches the CBOW assumption that context is a bag of words).
   - Trade-off: we lose positional information. If `the cat sat` and `cat the sat` are conceptually different, mean-pool can't tell. Concatenation would preserve order but blow up the input size and break order-invariance.

2. **Initial loss with vocab_size = 10000?**
   - Approximately `ln(10000) ≈ 9.21`. (Cross-entropy of the uniform distribution over V classes is `ln(V)`.)

3. **Why does `most_similar('good')` return `terrible`?**
   - Distributional similarity is not semantic identity. *good* and *terrible* both fit slots like *"the food was ___"*, so they share contexts and CBOW pulls their vectors together. CBOW learns **substitutability**, not sentiment. To separate antonyms you need a supervised signal (sentiment label) or a dedicated objective like supervised contrastive learning.

### Next up

**Notebook 6. Pretrained Embeddings (GloVe).** We reuse the CBOW class but initialize `nn.Embedding` from Stanford's GloVe vectors and compare frozen vs. fine-tuned.